# 03 — Model Training & Comparison

**SmartEnergy Predictor — PJK-GM096**

Notebook ini melatih 3 algoritma (XGBoost, Random Forest, Linear Regression) pada data konsumsi listrik dan membandingkan akurasi.

Untuk artifact production gunakan `python ml/train.py`. Notebook ini fokus pada eksplorasi & validasi.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

from ml.evaluate import evaluate, format_metrics_table
from ml.preprocessing import PreprocessingPipeline, TARGET_COL

df = pd.read_csv(ROOT / 'data' / 'processed' / 'household_consumption_clean.csv', parse_dates=['tanggal']).sort_values('tanggal').reset_index(drop=True)
print('Shape:', df.shape)

## 1. Chronological Split (no leakage)

In [ ]:
n_test = int(len(df) * 0.2)
df_train = df.iloc[:-n_test].copy()
df_test = df.iloc[-n_test:].copy()
print(f'Train: {len(df_train)}  Test: {len(df_test)}')
print(f'Train range: {df_train.tanggal.min().date()} -> {df_train.tanggal.max().date()}')
print(f'Test range : {df_test.tanggal.min().date()} -> {df_test.tanggal.max().date()}')

## 2. Preprocessing

In [ ]:
pipe = PreprocessingPipeline()
X_train_df, y_train_series = pipe.fit_transform(df_train)
X_train = X_train_df.values
y_train = y_train_series.values

X_test = pipe.transform(df_test).values
y_test = df_test[TARGET_COL].values
print('X_train:', X_train.shape, '  X_test:', X_test.shape)

## 3. XGBoost - RandomizedSearchCV + TimeSeriesSplit

Untuk demo cepat di notebook kita gunakan `n_iter=10`. Production script (`ml/train.py`) menggunakan `n_iter=25`.

In [ ]:
param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

base = xgb.XGBRegressor(objective='reg:squarederror', tree_method='hist', random_state=42, n_jobs=-1)
search = RandomizedSearchCV(
    estimator=base,
    param_distributions=param_grid,
    n_iter=10,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print(f'Best CV RMSE: {-search.best_score_:.4f}')
xgb_model = search.best_estimator_

## 4. Random Forest & Linear Regression (baselines)

In [ ]:
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42).fit(X_train, y_train)
lr = LinearRegression().fit(X_train, y_train)

metrics = {
    'XGBoost': evaluate(y_test, xgb_model.predict(X_test)),
    'RandomForest': evaluate(y_test, rf.predict(X_test)),
    'LinearRegression': evaluate(y_test, lr.predict(X_test)),
}
print(format_metrics_table(metrics))

## 5. Visualisasi Prediksi vs Aktual (XGBoost)

In [ ]:
import matplotlib.pyplot as plt

y_pred = xgb_model.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(y_test, y_pred, alpha=0.4, color='#10b981')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', label='y = x')
axes[0].set_xlabel('Actual kWh'); axes[0].set_ylabel('Predicted kWh')
axes[0].set_title('Prediksi vs Aktual (XGBoost)')
axes[0].legend()

residuals = y_test - y_pred
axes[1].hist(residuals, bins=40, color='#10b981', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Distribusi Residual')
axes[1].set_xlabel('residual (kWh)')

plt.tight_layout(); plt.show()

Target R² ≥ 0.85 dicapai oleh XGBoost. Lanjutkan ke `04_model_evaluation.ipynb` untuk evaluasi mendalam (feature importance, error analysis).